# Why do we program with classes?

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

This notebook is about **why**, not about how.

The syntax &mdash; `class`, `self`, constructors, inheritance &mdash; is covered in
[classes_intro](./classes_intro.ipynb), [classes](./classes.ipynb) and
[classes_inheritance](./classes_inheritance.ipynb). Here we only ask what problem
classes solve, and when they are the wrong answer.

## The question

You already know how to write functions. Functions are enough to compute anything.

So why does almost every serious scientific code &mdash; ROOT, Geant4, astropy, scikit-learn,
even `numpy` itself &mdash; hand you **objects** instead of functions?

The answer is not "because it is modern", and it is not "because it is elegant".
It is that four specific things become painful as a program grows, and classes make
three of them go away.

## Let us start from a problem

We want to simulate particles moving under a force. With functions alone:

In [ ]:
def step(x, v, m, force, dt):
    a = force / m
    return x + v * dt, v + a * dt

x, v, m = 0.0, 1.0, 2.0
for _ in range(3):
    x, v = step(x, v, m, force=-1.0, dt=0.1)
print(f"x = {x:.3f}, v = {v:.3f}")

This works. It is clear. For one particle and one afternoon, stop here &mdash;
you do not need a class.

The trouble starts when the program grows.

## Symptom 1: data that belongs together travels apart

`x`, `v` and `m` describe **one** particle, but the language does not know that.
Nothing stops us from mixing them up.

In [ ]:
# a second particle...
x2, v2, m2 = 5.0, -1.0, 3.0

# ...and a typo that Python is perfectly happy with
x2, v2 = step(x2, v, m2, force=-1.0, dt=0.1)   # used v, not v2
print(f"x2 = {x2:.3f}, v2 = {v2:.3f}   <- silently wrong")

No error, no warning, a wrong physics result. With three particles and five
quantities each you are carrying fifteen loose variables, and the chance of one of
these mistakes approaches certainty.

**A class binds the data that belongs together into one thing you cannot take apart by accident.**

In [ ]:
class Particle:
    def __init__(self, x, v, m):
        self.x, self.v, self.m = x, v, m

    def step(self, force, dt):
        self.v += (force / self.m) * dt
        self.x += self.v * dt

p = Particle(5.0, -1.0, 3.0)
p.step(force=-1.0, dt=0.1)
print(f"x = {p.x:.3f}, v = {p.v:.3f}")

Now there is no way to pass *this* particle's position with *that* particle's velocity.
The mistake has become unwriteable.

## Symptom 2: things that must always be true

A mass is positive. A direction vector has unit length. A probability lies in [0, 1].
These are **invariants**: statements that must hold at every moment.

With loose variables, every piece of code that touches the data must remember to
check. With a class, there is **one place** where it is checked.

In [ ]:
class Particle:
    def __init__(self, x, v, m):
        if m <= 0:
            raise ValueError(f"mass must be positive, got {m}")
        self.x, self.v, self.m = x, v, m

    def step(self, force, dt):
        self.v += (force / self.m) * dt
        self.x += self.v * dt

try:
    Particle(0.0, 1.0, m=-2.0)
except ValueError as e:
    print("caught:", e)

The object cannot exist in an invalid state. That is worth more than it sounds:
most long debugging sessions are spent finding out *where* the data went wrong,
not *what* went wrong.

## Symptom 3: the same operation on different things

This is the one that actually matters.

Suppose the force can come from several models. Written procedurally, the simulation
loop has to know about every one of them:

In [ ]:
def force_of(kind, x):
    if kind == "spring":
        return -1.0 * x
    elif kind == "gravity":
        return -9.8
    elif kind == "drag":
        return -0.5 * x
    else:
        raise ValueError(kind)

print([round(force_of(k, 2.0), 2) for k in ("spring", "gravity", "drag")])

Every new model means editing this function. And `force_of` is not the only place:
the same `if/elif` chain reappears wherever the model matters. Add a fourth force
and you must find all of them.

### The same thing with objects

Each model is an object that knows how to compute its own force.

In [ ]:
class Spring:
    def __init__(self, k): self.k = k
    def force(self, x):    return -self.k * x

class Gravity:
    def __init__(self, g, m): self.g, self.m = g, m
    def force(self, x):       return -self.m * self.g

class Drag:
    def __init__(self, b): self.b = b
    def force(self, x):    return -self.b * x

for model in (Spring(1.0), Gravity(9.8, 1.0), Drag(0.5)):
    print(f"{type(model).__name__:8s} -> {model.force(2.0):6.2f}")

### And now the point

The code that *uses* the models never mentions any of them:

In [ ]:
def simulate(particle, model, dt, n):
    for _ in range(n):
        particle.step(model.force(particle.x), dt)
    return particle.x

print(f"spring : {simulate(Particle(1.0, 0.0, 1.0), Spring(1.0),  0.01, 100):.4f}")
print(f"drag   : {simulate(Particle(1.0, 0.0, 1.0), Drag(0.5),    0.01, 100):.4f}")

Adding a fourth force model requires **no change at all** to `simulate`.
It only has to have a method called `force`.

> This is the real argument for classes: **the code that uses a thing does not
> need to know what the thing is** &mdash; only what it can do.

That is what people mean by *polymorphism*, and it is why every simulation framework
you will meet is built this way.

## Symptom 4: you have been using classes all along

You do not get to opt out. In Python, everything is an object.

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
print(type(a))
print(a.mean(), a.sum(), a.shape)     # methods and attributes of an object

s = "hello"
print(type(s), s.upper())

`a.mean()` is a method call on an instance of `numpy.ndarray`. `s.upper()` is a
method call on an instance of `str`.

So the question is never *"should I use objects?"* &mdash; you already do.
It is only *"should I write a new kind of object myself?"*.

## So what is a class, really?

Three things bound together:

| | |
|---|---|
| **State** | the data that describes one instance (`x`, `v`, `m`) |
| **Behaviour** | what you can do to it (`step`, `force`) |
| **A promise** | the *interface*: any object with a `force(x)` method can be used by `simulate` |

The promise is the part people forget, and the part that does the work.

## The four words people will throw at you

| Word | What it actually means | Honest weight |
|---|---|---|
| **Encapsulation** | keep the data and the code that maintains it together | useful |
| **Abstraction** | expose what it does, hide how | useful |
| **Polymorphism** | different objects, same interface | **the reason classes exist** |
| **Inheritance** | a class built on another | overused; see below |

Textbooks give the four equal billing. They are not equal.

### A warning about inheritance

Inheritance is the most abused idea in object-oriented programming. A deep hierarchy
(`Detector` &rarr; `GaseousDetector` &rarr; `TPC` &rarr; `HighPressureTPC` &rarr; ...) is
very hard to change later, because every class depends on all its ancestors.

The rule of thumb, which the Python community follows and you should too:

> **Prefer composition to inheritance.** An object that *has* a `Geometry` and a
> `ReadOut` is easier to change than one that *is* a subclass of five things.

Use inheritance when you genuinely mean "is a kind of", and keep it one level deep
if you can.

## When *not* to write a class

Python is not Java. There is no rule that says code must live in a class,
and a class that is not carrying state is just a function wearing a costume.

In [ ]:
# An anti-pattern: no state, one method
class EnergyCalculator:
    def compute(self, m):
        return m * 299792458**2

print(EnergyCalculator().compute(1.0))

# The same thing, better
def energy(m):
    return m * 299792458**2

print(energy(1.0))

**If it has no state and one method, it is a function. Write the function.**

### Lighter alternatives

Often you want to group data, but not behaviour. Python has cheaper tools than a full class:

In [ ]:
from dataclasses import dataclass

@dataclass
class Measurement:
    value: float
    error: float
    units: str = "MeV"

m = Measurement(125.3, 0.4)
print(m)                       # a readable repr, for free
print(m.value + m.error)

| Tool | Use it when |
|---|---|
| a plain **function** | no state to keep |
| a **tuple** or **dict** | throwaway grouping, used in one place |
| `@dataclass` / `NamedTuple` | data with names, little or no behaviour |
| a **class** | state *and* behaviour, invariants, or several implementations of one interface |
| a **numpy array** | many identical things &mdash; see below |

## The cost, in physics

A class per particle is fine for ten particles. For ten million it costs you more
than an order of magnitude: each object carries its own memory overhead, and the loop
runs in Python instead of in C. The gap grows with the work done per step.

Measure it yourself &mdash; the numbers below depend on your machine.

In [ ]:
import numpy as np, time

N = 500_000

class P:
    def __init__(self, x, v): self.x, self.v = x, v

ps = [P(0.0, 1.0) for _ in range(N)]
t0 = time.perf_counter()
for p in ps:
    p.x += p.v * 0.01
t_obj = time.perf_counter() - t0

x = np.zeros(N); v = np.ones(N)
t0 = time.perf_counter()
x += v * 0.01
t_arr = time.perf_counter() - t0

print(f"one object per particle : {t_obj*1e3:7.1f} ms")
print(f"numpy arrays            : {t_arr*1e3:7.1f} ms")
print(f"speed-up                : {t_obj/t_arr:7.0f}x")

So the right design for a physics code is usually **not** a million small objects.
It is a *few* objects, each holding **arrays**: one `Event` with arrays of hits,
one `Detector` with arrays of channels.

Objects for the structure; arrays for the numbers.

## When to reach for a class: a checklist

Write a class when you can answer *yes* to at least one of these:

1. Do several pieces of data always travel together?
2. Is there something that must always be true about them?
3. Will there be **more than one implementation** of the same idea,
   and do you want the rest of the code not to care which?
4. Does the thing have a lifetime &mdash; created, used many times, changed, closed?

If the answer to all four is *no*, write a function.

## Summary

- Classes are not about elegance; they solve four concrete problems.
- **Binding data together** removes a whole family of silent bugs.
- **Invariants** get one place to live, instead of being everyone's responsibility.
- **Polymorphism is the real payoff**: code that uses an object need not know what it is.
- You are already using objects constantly, whether or not you write any.
- A class with no state and one method should be a function.
- In numerical work: objects for the structure, **arrays for the numbers**.

Next: the syntax, in [classes_intro](./classes_intro.ipynb).

## Exercises

**1.** The code below is procedural. Decide what should become a class, what should
stay a function, and what should be a `dataclass`. Justify each choice with the
checklist above &mdash; do not write the code yet.

```python
def read_spectrum(path): ...
def calibrate(counts, slope, offset): ...
def subtract_background(counts, bkg): ...
def fit_peak(energies, counts, e_min, e_max): ...
def plot(energies, counts, title): ...
```

**2.** Take the `simulate` function of this notebook and add a fourth force model
(a constant push, `force(x) = F0`). How many lines of existing code did you have
to change? Now do the same for the `force_of` version with the `if/elif` chain.

**3.** Find one class in `numpy` or `matplotlib` whose interface you use often,
and name three different objects that satisfy it.

**4.** Argue *against* using a class for a Monte Carlo generator that produces
10 million events. What would you use instead?